# iTransform Context Forecasting

<!-- Commented sorted notebook -->
Purpose: train a discriminative forecasting baseline using the same frequency, horizon, context-window, customer split, and validation-export format as the generative forecasting notebooks.

The output is a checkpoint plus validation metrics/plots that can be compared with the generative models.


In [ ]:

MODEL_DIR_NAME = "iTransform"
MODEL_LABEL = "iTransformer"
MODEL_TAG = "itransformer"

from pathlib import Path
from datetime import datetime
import json
import math
import re
import sys
import warnings
import zipfile
from xml.sax.saxutils import escape

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=r".*'T' is deprecated.*", category=FutureWarning)

FREQ = "30min"
PRIMARY_HORIZON = "24h"
FORECAST_HORIZONS = {"24h": pd.Timedelta("24h"), "7d": pd.Timedelta("7d"), "28d": pd.Timedelta("28d")}
ALLOWED_FREQS = {"30min", "1H", "2H", "3H", "6H"}

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
MAX_STEPS = 300
EVAL_WINDOWS = 64
CONTEXT_HORIZONS = {"24h": pd.Timedelta("2d"), "7d": pd.Timedelta("7d"), "28d": pd.Timedelta("7d")}
TRAIN_STRIDE_MULTIPLIER = 1
WINDOWS_BATCH_SIZE = 128
NUM_LOADER_WORKERS = 2 if sys.platform != "win32" else 0
QUANTILES = [0.1, 0.5, 0.9]
ALPHA_PI = 0.20
BINS = 80

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

STATIC_BASE_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]
STATIC_CAT_CANDIDATES = [
    "NEAREST_STATION_NO", "STATION_NO", "station_id", "StationID",
    "TrialRegion", "TRIAL_REGION", "TrialRegionID", "trial_region_id",
]
PAST_COV_COLS = [
    "Temperature", "CDD", "HDD", "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6", "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6", "temperature_lag_48", "temperature_lag_144",
]
FUTURE_COV_COLS = [
    "HourSin", "HourCos", "HalfHourSin", "HalfHourCos",
    "WeekdaySin", "WeekdayCos", "MonthSin", "MonthCos",
    "Summer", "Fall", "Winter", "Spring",
]


# Keeps pandas frequency strings explicit so 30min/1H/2H/3H experiments resample consistently.
def pandas_freq(freq):
    return freq.replace("H", "h")


# Converts a human horizon label into target-window steps at the selected data frequency.
def seq_len_for_horizon(primary_horizon, freq):
    if primary_horizon not in FORECAST_HORIZONS:
        raise ValueError(f"Unsupported horizon {primary_horizon}. Use one of {sorted(FORECAST_HORIZONS)}")
    if freq not in ALLOWED_FREQS:
        raise ValueError(f"Unsupported freq {freq}. Use one of {sorted(ALLOWED_FREQS)}")
    steps = FORECAST_HORIZONS[primary_horizon] / pd.to_timedelta(pandas_freq(freq))
    if not float(steps).is_integer():
        raise ValueError(f"Horizon {primary_horizon} is not divisible by freq {freq}")
    return int(steps)


# Uses fixed calendar-duration context windows so context length is comparable across frequencies.
def context_len_for_horizon(primary_horizon, freq):
    if primary_horizon not in CONTEXT_HORIZONS:
        raise ValueError(f"Unsupported context horizon {primary_horizon}. Use one of {sorted(CONTEXT_HORIZONS)}")
    steps = CONTEXT_HORIZONS[primary_horizon] / pd.to_timedelta(pandas_freq(freq))
    if not float(steps).is_integer():
        raise ValueError(f"Context horizon {primary_horizon} is not divisible by freq {freq}")
    return int(steps)

SEQ_LEN = seq_len_for_horizon(PRIMARY_HORIZON, FREQ)
INPUT_LEN = context_len_for_horizon(PRIMARY_HORIZON, FREQ)
TRAIN_STRIDE = max(1, SEQ_LEN * TRAIN_STRIDE_MULTIPLIER)


def find_models_dir():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([base, base / "models"])
    for base in candidates:
        if (base / "data_with_weather.pickle").exists() or (base / "data_min.parquet").exists():
            return base
    raise FileNotFoundError("Could not find data_with_weather.pickle or data_min.parquet.")


MODELS_DIR = find_models_dir()
MODEL_DIR = MODELS_DIR / MODEL_DIR_NAME
print(f"MODEL_DIR={MODEL_DIR}")
print(f"FREQ={FREQ} | horizon={PRIMARY_HORIZON} | SEQ_LEN={SEQ_LEN} | INPUT_LEN={INPUT_LEN} | TRAIN_STRIDE={TRAIN_STRIDE}")



In [ ]:

# Cyclical sin/cos calendar encoding preserves periodic time distance; source: https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
def add_calendar_covariates(df, time_values):
    dt = pd.to_datetime(time_values)
    hour = dt.dt.hour.astype("float32")
    minute = dt.dt.minute.astype("float32")
    half_hour = hour * 2 + (minute // 30)
    weekday = dt.dt.weekday.astype("float32")
    month = dt.dt.month.astype("float32")
    df["HourSin"] = np.sin(2.0 * np.pi * hour / 24.0).astype("float32")
    df["HourCos"] = np.cos(2.0 * np.pi * hour / 24.0).astype("float32")
    df["HalfHourSin"] = np.sin(2.0 * np.pi * half_hour / 48.0).astype("float32")
    df["HalfHourCos"] = np.cos(2.0 * np.pi * half_hour / 48.0).astype("float32")
    df["WeekdaySin"] = np.sin(2.0 * np.pi * weekday / 7.0).astype("float32")
    df["WeekdayCos"] = np.cos(2.0 * np.pi * weekday / 7.0).astype("float32")
    df["MonthSin"] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    df["MonthCos"] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    df["Summer"] = month.isin([12, 1, 2]).astype("float32")
    df["Fall"] = month.isin([3, 4, 5]).astype("float32")
    df["Winter"] = month.isin([6, 7, 8]).astype("float32")
    df["Spring"] = month.isin([9, 10, 11]).astype("float32")
    return df


# Encodes string categories as stable integer IDs before embedding or library ingestion.
def static_cat_code_col(col):
    return "cat_" + re.sub(r"[^A-Za-z0-9]+", "_", col).strip("_")


def load_energy_dataframe():
    pickle_path = MODELS_DIR / "data_with_weather.pickle"
    parquet_path = MODELS_DIR / "data_min.parquet"
    if pickle_path.exists():
        df = pd.read_pickle(pickle_path)
    elif parquet_path.exists():
        df = pd.read_parquet(parquet_path)
    else:
        raise FileNotFoundError(f"No data file found in {MODELS_DIR}")
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
    df = add_calendar_covariates(df, df[TIME_COL])

    if "Temperature" in df.columns:
        temp = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
        if "CDD" not in df.columns:
            df["CDD"] = np.maximum(temp - 18.0, 0.0).astype("float32")
        if "HDD" not in df.columns:
            df["HDD"] = np.maximum(18.0 - temp, 0.0).astype("float32")

    static_cat_code_cols = []
    for col in STATIC_CAT_CANDIDATES:
        if col in df.columns:
            code_col = static_cat_code_col(col)
            codes, _ = pd.factorize(df[col].astype("string").fillna("__missing__"), sort=True)
            df[code_col] = codes.astype("float32")
            static_cat_code_cols.append(code_col)

    numeric_cols = [TARGET_COL] + [
        c for c in PAST_COV_COLS + FUTURE_COV_COLS + STATIC_BASE_COLS + static_cat_code_cols if c in df.columns
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype("float32")

    agg = {TARGET_COL: "sum"}
    for col in PAST_COV_COLS + FUTURE_COV_COLS:
        if col in df.columns and col not in agg:
            agg[col] = "mean"
    for col in STATIC_BASE_COLS + static_cat_code_cols:
        if col in df.columns:
            agg[col] = "first"
    df = (
        df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])
          .set_index(TIME_COL)
          .groupby(HOUSEHOLD_ID_COL)
          .resample(pandas_freq(FREQ))
          .agg(agg)
          .reset_index()
    )
    df = add_calendar_covariates(df, df[TIME_COL])
    return df.sort_values([HOUSEHOLD_ID_COL, TIME_COL]).reset_index(drop=True), static_cat_code_cols


# Splits by customer ID, not by row, to avoid leakage between train and validation households.
def split_customers(df):
    rng = np.random.default_rng(SEED)
    customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
    n_val = max(1, int(len(customers) * VAL_FRAC))
    val_customers = set(rng.choice(customers, size=n_val, replace=False).tolist())
    train_customers = set(customers.tolist()) - val_customers
    return train_customers, val_customers


df, STATIC_CAT_CODE_COLS = load_energy_dataframe()
STATIC_COLS = STATIC_BASE_COLS + STATIC_CAT_CODE_COLS
train_customers, val_customers = split_customers(df)
print(f"Dataframe shape: {df.shape}")
print(f"Customers total={len(train_customers) + len(val_customers)} | train={len(train_customers)} | val={len(val_customers)}")



In [ ]:

# Pinball loss evaluates quantile forecasts; source: https://en.wikipedia.org/wiki/Quantile_regression
def pinball_loss(y_true, y_pred, q):
    err = np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)
    return float(np.nanmean(np.maximum(q * err, (q - 1.0) * err)))


# Drops NaN/Inf values before histogram, QQ, and residual calculations.
def finite_flat(arr):
    x = np.asarray(arr, dtype=float).reshape(-1)
    return x[np.isfinite(x)]


# Autocorrelation checks whether forecasts preserve temporal dependence; source: https://en.wikipedia.org/wiki/Autocorrelation
def autocorr_1d(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    x = np.nan_to_num(x, nan=0.0)
    denom = float(np.dot(x, x) + 1e-12)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.asarray(vals)


def mean_acf(arr):
    y = np.asarray(arr, dtype=float)[:, :, 0]
    max_lag = min(200, y.shape[1] - 1)
    if max_lag < 1:
        return np.ones(1)
    return np.nanmean([autocorr_1d(row, max_lag) for row in y], axis=0)


def power_spectrum_1d(x):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    ps = np.abs(np.fft.rfft(x)) ** 2
    return ps / (ps.sum() + 1e-12)


# Approximate DTW downsamples long windows before dynamic-programming alignment; source: https://en.wikipedia.org/wiki/Dynamic_time_warping
def approximate_dtw(a, b, max_points=256):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if len(a) > max_points:
        idx = np.linspace(0, len(a) - 1, max_points).round().astype(int)
        a = a[idx]
        b = b[idx]
    n, m = len(a), len(b)
    prev = np.full(m + 1, np.inf)
    curr = np.full(m + 1, np.inf)
    prev[0] = 0.0
    for i in range(1, n + 1):
        curr[0] = np.inf
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = abs(ai - b[j - 1])
            curr[j] = cost + min(prev[j], curr[j - 1], prev[j - 1])
        prev, curr = curr, prev
    return float(prev[m])


def sqrtm_psd(mat):
    mat = (mat + mat.T) / 2.0
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def feature_matrix(arr):
    y = np.asarray(arr, dtype=float)[:, :, 0]
    acf1, acf2, acf3 = [], [], []
    for row in y:
        acf = autocorr_1d(row, min(3, len(row) - 1))
        acf1.append(acf[1] if len(acf) > 1 else 0.0)
        acf2.append(acf[2] if len(acf) > 2 else 0.0)
        acf3.append(acf[3] if len(acf) > 3 else 0.0)
    return np.column_stack([
        np.nanmean(y, axis=1), np.nanstd(y, axis=1), np.nanmin(y, axis=1), np.nanmax(y, axis=1),
        np.nanquantile(y, 0.10, axis=1), np.nanquantile(y, 0.50, axis=1), np.nanquantile(y, 0.90, axis=1),
        np.nansum(y, axis=1), np.nanargmax(y, axis=1) / max(1, y.shape[1] - 1),
        np.asarray(acf1), np.asarray(acf2), np.asarray(acf3),
    ])


# Feature Fréchet distance mirrors FID-style Gaussian distance on summary features; source: https://arxiv.org/abs/1706.08500
def frechet_feature_distance(real, fake):
    xr = feature_matrix(real)
    xf = feature_matrix(fake)
    mu_r, mu_f = np.nanmean(xr, axis=0), np.nanmean(xf, axis=0)
    cov_r = np.cov(np.nan_to_num(xr).T) + np.eye(xr.shape[1]) * 1e-6
    cov_f = np.cov(np.nan_to_num(xf).T) + np.eye(xf.shape[1]) * 1e-6
    sqrt_r = sqrtm_psd(cov_r)
    covmean = sqrtm_psd(sqrt_r @ cov_f @ sqrt_r)
    value = float(np.sum((mu_r - mu_f) ** 2) + np.trace(cov_r + cov_f - 2.0 * covmean))
    return max(value, 0.0)


# Histogram KL divergence compares real and generated marginal kWh distributions; source: https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence
def kl_histogram(real, fake):
    real_flat = finite_flat(real)
    fake_flat = finite_flat(fake)
    lo, hi = float(np.nanmin(real_flat)), float(np.nanmax(real_flat))
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, BINS + 1)
    p, _ = np.histogram(real_flat, bins=edges, density=False)
    q, _ = np.histogram(fake_flat, bins=edges, density=False)
    eps = 1e-8
    p = p.astype(float) + eps
    q = q.astype(float) + eps
    p = p / p.sum()
    q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))


# Shared point/probabilistic metrics include CRPS-style quantile scoring; source: https://en.wikipedia.org/wiki/Continuous_ranked_probability_score
def compute_metrics(y_real, q10, q50, q90):
    y_real = np.asarray(y_real, dtype=float)
    q10 = np.asarray(q10, dtype=float)
    q50 = np.asarray(q50, dtype=float)
    q90 = np.asarray(q90, dtype=float)
    low = np.minimum(q10, q90)
    high = np.maximum(q10, q90)
    err = q50 - y_real
    abs_err = np.abs(err)
    width = high - low
    below = y_real < low
    above = y_real > high
    winkler = width + (2.0 / ALPHA_PI) * (low - y_real) * below + (2.0 / ALPHA_PI) * (y_real - high) * above
    acf_error = float(np.nanmean(np.abs(mean_acf(y_real) - mean_acf(q50))))
    n_dtw = min(32, y_real.shape[0])
    dtw_mean = float(np.nanmean([approximate_dtw(y_real[i, :, 0], q50[i, :, 0]) for i in range(n_dtw)]))
    scale = float(np.nanmean(y_real) + 1e-6)
    peak_scale = float(np.nanmean(np.nanmax(y_real[:, :, 0], axis=1)) + 1e-6)
    metrics = {
        # KL histogram estimates marginal distribution mismatch from binned real/forecast kWh; source: https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence
        "KL_hist": kl_histogram(y_real, q50),
        "DTW_mean": dtw_mean,
        "FTSD": frechet_feature_distance(y_real, q50),
        "MAE_median": float(np.nanmean(abs_err)),
        "RMSE_mean": float(np.sqrt(np.nanmean(err ** 2))),
        "PeakMAE": float(np.nanmean(np.abs(np.nanmax(q50[:, :, 0], axis=1) - np.nanmax(y_real[:, :, 0], axis=1)))),
        # CRPS is approximated from quantile pinball losses; source: https://en.wikipedia.org/wiki/Continuous_ranked_probability_score
        "CRPS": float(2.0 * np.nanmean([pinball_loss(y_real, low, 0.1), pinball_loss(y_real, q50, 0.5), pinball_loss(y_real, high, 0.9)])),
        # QuantileLoss averages pinball losses at the requested forecast quantiles; source: https://en.wikipedia.org/wiki/Quantile_regression
        "QuantileLoss": float(np.nanmean([pinball_loss(y_real, low, 0.1), pinball_loss(y_real, q50, 0.5), pinball_loss(y_real, high, 0.9)])),
        # Winkler interval score rewards narrow prediction intervals but penalises missed coverage; source: https://epiforecasts.io/scoringutils/articles/scoring-rules.html
        "Winkler": float(np.nanmean(winkler)),
        "Coverage": float(np.nanmean((y_real >= low) & (y_real <= high))),
        "ACF_error": acf_error,
    }
    # CompositeScore is a project-specific tuning heuristic, not a standard literature metric.
    metrics["CompositeScore"] = float(np.nanmean([
        metrics["MAE_median"] / scale,
        metrics["RMSE_mean"] / scale,
        metrics["PeakMAE"] / peak_scale,
        metrics["CRPS"] / scale,
        metrics["QuantileLoss"] / scale,
        metrics["Winkler"] / scale,
        metrics["ACF_error"],
        metrics["KL_hist"],
        metrics["DTW_mean"] / (max(1, SEQ_LEN) * scale),
        min(metrics["FTSD"], 1e6) / (1.0 + min(metrics["FTSD"], 1e6)),
    ]))
    return metrics


# Writes a minimal Excel workbook without adding an openpyxl dependency.
def write_metrics_xlsx(path, rows):
    def as_value(value):
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, (list, tuple, np.ndarray)):
            return json.dumps(np.asarray(value).tolist())
        if value is None:
            return ""
        if isinstance(value, float) and not math.isfinite(value):
            return str(value)
        return value

    def col_name(idx):
        name = ""
        while idx:
            idx, rem = divmod(idx - 1, 26)
            name = chr(65 + rem) + name
        return name

    def cell_xml(row_idx, col_idx, value):
        ref = f"{col_name(col_idx)}{row_idx}"
        value = as_value(value)
        if isinstance(value, bool):
            return f'<c r="{ref}" t="b"><v>{1 if value else 0}</v></c>'
        if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value)):
            return f'<c r="{ref}"><v>{value}</v></c>'
        return f'<c r="{ref}" t="inlineStr"><is><t>{escape(str(value))}</t></is></c>'

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sheet_rows = []
    for r_idx, row in enumerate(rows, start=1):
        cells = "".join(cell_xml(r_idx, c_idx, value) for c_idx, value in enumerate(row, start=1))
        sheet_rows.append(f'<row r="{r_idx}">{cells}</row>')
    worksheet = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main"><sheetData>' + "".join(sheet_rows) + '</sheetData></worksheet>'
    workbook = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships"><sheets><sheet name="metrics" sheetId="1" r:id="rId1"/></sheets></workbook>'
    content_types = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/><Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/></Types>'
    root_rels = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>'
    workbook_rels = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/></Relationships>'
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("[Content_Types].xml", content_types)
        zf.writestr("_rels/.rels", root_rels)
        zf.writestr("xl/workbook.xml", workbook)
        zf.writestr("xl/_rels/workbook.xml.rels", workbook_rels)
        zf.writestr("xl/worksheets/sheet1.xml", worksheet)


def save_fig(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return str(path)


# Creates the common diagnostic plot set used for visual model comparison.
def save_common_plots(out_dir, y_real, q10, q50, q90):
    out_dir = Path(out_dir)
    plot_paths = []
    real_y = np.asarray(y_real, dtype=float)[:, :, 0]
    fake_y = np.asarray(q50, dtype=float)[:, :, 0]
    low_y = np.minimum(np.asarray(q10, dtype=float)[:, :, 0], np.asarray(q90, dtype=float)[:, :, 0])
    high_y = np.maximum(np.asarray(q10, dtype=float)[:, :, 0], np.asarray(q90, dtype=float)[:, :, 0])
    t = np.arange(real_y.shape[1])
    real_flat = finite_flat(real_y)
    fake_flat = finite_flat(fake_y)
    lo, hi = float(real_flat.min()), float(real_flat.max())
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, BINS + 1)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(t, np.nanmean(real_y, axis=0), label="Real mean", linewidth=2)
    real_lo, real_hi = np.nanquantile(real_y, [0.10, 0.90], axis=0)
    ax.fill_between(t, real_lo, real_hi, alpha=0.18, label="Real 10-90% band")
    ax.plot(t, np.nanmean(fake_y, axis=0), label="Forecast median mean", linewidth=2)
    ax.fill_between(t, np.nanmean(low_y, axis=0), np.nanmean(high_y, axis=0), alpha=0.18, label="Mean q10-q90 interval")
    ax.set_title(f"Multi-window Real vs Forecast Profile | {MODEL_LABEL}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("kWh")
    ax.legend(loc="upper right")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "01_multi_window_profile_bands.png"))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(real_flat, bins=edges, density=True, alpha=0.6, label="Real")
    ax.hist(fake_flat, bins=edges, density=True, alpha=0.6, label="Forecast median")
    ax.set_title(f"All-window kWh Distribution | {MODEL_LABEL}")
    ax.set_xlabel("kWh")
    ax.set_ylabel("Density")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "02_probability_distribution.png"))

    q = np.linspace(0.01, 0.99, 300)
    real_q = np.quantile(real_flat, q)
    fake_q = np.quantile(fake_flat, q)
    mx = float(max(real_q.max(), fake_q.max(), 1e-6))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(real_q, fake_q, marker=".", linestyle="none", alpha=0.6)
    ax.plot([0, mx], [0, mx], linewidth=1)
    ax.set_title(f"All-window QQ Plot | {MODEL_LABEL}")
    ax.set_xlabel("Real quantiles")
    ax.set_ylabel("Forecast quantiles")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "03_qq_plot.png"))

    acf_real = mean_acf(y_real)
    acf_fake = mean_acf(q50)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(acf_real, label="Real ACF")
    ax.plot(acf_fake, label="Forecast ACF")
    ax.set_title(f"Autocorrelation | {MODEL_LABEL}")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "04_autocorrelation.png"))

    ps_real = np.mean([power_spectrum_1d(row) for row in real_y], axis=0)
    ps_fake = np.mean([power_spectrum_1d(row) for row in fake_y], axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ps_real, label="Real power spectrum")
    ax.plot(ps_fake, label="Forecast power spectrum")
    ax.set_title(f"Power Spectrum | {MODEL_LABEL}")
    ax.set_xlabel("Frequency bin")
    ax.set_ylabel("Normalized power")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "05_power_spectrum.png"))

    resid = fake_y - real_y
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(finite_flat(resid), bins=100, density=True, alpha=0.8)
    ax.set_title(f"Residual Distribution: Forecast - Real | {MODEL_LABEL}")
    ax.set_xlabel("kWh error")
    ax.set_ylabel("Density")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "06_residual_distribution.png"))

    true_vals = finite_flat(real_y)
    err_abs = finite_flat(np.abs(resid))
    n = min(true_vals.shape[0], err_abs.shape[0])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(true_vals[:n:50], err_abs[:n:50], alpha=0.3, s=5)
    ax.set_title(f"|Error| vs True kWh | {MODEL_LABEL}")
    ax.set_xlabel("True kWh")
    ax.set_ylabel("|Forecast - Real|")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "07_error_vs_true.png"))

    abs_err = np.abs(resid)
    window_mae = np.nanmean(abs_err, axis=1)
    window_rmse = np.sqrt(np.nanmean(resid ** 2, axis=1))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(window_mae[np.isfinite(window_mae)], bins=60, alpha=0.7, label="Window MAE")
    ax.hist(window_rmse[np.isfinite(window_rmse)], bins=60, alpha=0.5, label="Window RMSE")
    ax.set_title(f"Window-level Error Distribution | {MODEL_LABEL}")
    ax.set_xlabel("kWh error")
    ax.set_ylabel("Window count")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "08_window_error_distribution.png"))

    real_mean_w = np.nanmean(real_y, axis=1)
    fake_mean_w = np.nanmean(fake_y, axis=1)
    real_peak_w = np.nanmax(real_y, axis=1)
    fake_peak_w = np.nanmax(fake_y, axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, x_vals, y_vals, label in [
        (axes[0], real_mean_w, fake_mean_w, "Window mean kWh"),
        (axes[1], real_peak_w, fake_peak_w, "Window peak kWh"),
    ]:
        ax.scatter(x_vals, y_vals, alpha=0.35, s=12)
        finite = np.isfinite(x_vals) & np.isfinite(y_vals)
        if np.any(finite):
            lim = float(max(np.nanmax(x_vals[finite]), np.nanmax(y_vals[finite]), 1e-6))
            ax.plot([0, lim], [0, lim], linewidth=1)
        ax.set_title(label)
        ax.set_xlabel("Real")
        ax.set_ylabel("Forecast")
    fig.suptitle(f"Window Summary Agreement | {MODEL_LABEL}")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "09_window_summary_scatter.png"))

    med_ae = np.nanmedian(abs_err, axis=0)
    lo_ae, hi_ae = np.nanquantile(abs_err, [0.10, 0.90], axis=0)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, med_ae, label="Median absolute error")
    ax.fill_between(t, lo_ae, hi_ae, alpha=0.2, label="10-90% absolute error band")
    ax.set_title(f"Error by Timestep Across Windows | {MODEL_LABEL}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("kWh absolute error")
    ax.legend()
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "10_error_by_timestep.png"))

    order = np.argsort(real_mean_w)
    vmax = np.nanquantile(np.abs(resid), 0.99)
    vmax = float(vmax if np.isfinite(vmax) and vmax > 0 else 1.0)
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(resid[order], aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Residual Heatmap Across Windows | {MODEL_LABEL}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Evaluation windows sorted by real mean kWh")
    fig.colorbar(im, ax=ax, label="Forecast - Real kWh")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "11_residual_heatmap.png"))

    coverage_t = np.nanmean((real_y >= low_y) & (real_y <= high_y), axis=0)
    width_t = np.nanmean(high_y - low_y, axis=0)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(t, coverage_t)
    axes[0].axhline(1.0 - ALPHA_PI, linewidth=1, linestyle="--")
    axes[0].set_title(f"80% Prediction Interval Coverage by Timestep | {MODEL_LABEL}")
    axes[0].set_ylabel("Coverage")
    axes[1].plot(t, width_t)
    axes[1].set_title("Mean Prediction Interval Width")
    axes[1].set_xlabel("Timestep")
    axes[1].set_ylabel("kWh")
    fig.tight_layout()
    plot_paths.append(save_fig(fig, out_dir / "12_interval_coverage_by_timestep.png"))
    return plot_paths


# Writes checkpoint metadata, metrics.xlsx, and shared diagnostic plots for baseline models.
def export_results(checkpoint_name, config, y_real, q10, q50, q90, output_root):
    metrics = compute_metrics(y_real, q10, q50, q90)
    out_dir = Path(output_root) / checkpoint_name
    rows = [
        ["field", "value"],
        ["model", MODEL_LABEL],
        ["checkpoint_name", checkpoint_name],
        ["checkpoint_path", config.get("CHECKPOINT_PATH", "")],
        ["exported_at", datetime.now().isoformat(timespec="seconds")],
    ]
    for key, value in config.items():
        rows.append([key, value])
    rows.append(["quantiles", QUANTILES])
    rows.append(["interval_percent", (1.0 - ALPHA_PI) * 100.0])
    rows.append(["", ""])
    rows.append(["metric", "value"])
    for key in [
        "KL_hist", "DTW_mean", "FTSD", "MAE_median", "RMSE_mean", "PeakMAE",
        "CRPS", "QuantileLoss", "Winkler", "Coverage", "ACF_error", "CompositeScore",
    ]:
        rows.append([key, metrics[key]])
    metrics_path = out_dir / "metrics.xlsx"
    write_metrics_xlsx(metrics_path, rows)
    plot_paths = save_common_plots(out_dir / "plots", y_real, q10, q50, q90)
    print("=" * 60)
    print(f"[{MODEL_LABEL}] exported: {out_dir}")
    for key, value in metrics.items():
        print(f"{key}: {value:.6f}")
    print("Saved metrics:", metrics_path)
    print("Saved plots:", len(plot_paths))
    return out_dir, metrics_path, plot_paths, metrics



In [ ]:

import torch
# NeuralForecast runner used for iTransformer; docs: https://nixtlaverse.nixtla.io/neuralforecast/
# Implementation codebase for NeuralForecast training/inference wrappers: https://github.com/Nixtla/neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss
# iTransformer baseline model; paper: https://arxiv.org/abs/2310.06625
# Official iTransformer implementation reference: https://github.com/thuml/iTransformer
from neuralforecast.models import iTransformer


# Keeps GPU/precision/logger settings local to the library trainer call.
def trainer_kwargs():
    kwargs = {
        "enable_checkpointing": False,
        "logger": False,
        "enable_progress_bar": True,
        "gradient_clip_val": 1.0,
    }
    if torch.cuda.is_available():
        kwargs.update({"accelerator": "gpu", "devices": 1, "precision": "16-mixed"})
    else:
        kwargs.update({"accelerator": "cpu", "devices": 1})
    return kwargs


# Keeps dataloader worker settings explicit for repeatable server-side training.
def dataloader_kwargs():
    kwargs = {"num_workers": NUM_LOADER_WORKERS}
    if NUM_LOADER_WORKERS > 0:
        kwargs["persistent_workers"] = True
    if torch.cuda.is_available():
        kwargs["pin_memory"] = True
    return kwargs


# Converts customer windows into NeuralForecast long-format dataframes.
def build_neuralforecast_frames():
    past_cov_cols = [c for c in PAST_COV_COLS if c in df.columns]
    future_cov_cols = [c for c in FUTURE_COV_COLS if c in df.columns]
    static_cols = [c for c in STATIC_COLS if c in df.columns]
    freq_alias = pandas_freq(FREQ)
    frames = []
    static_rows = []

    for customer_id, g in df.groupby(HOUSEHOLD_ID_COL):
        g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL], keep="last")
        if g.empty:
            continue
        idx = pd.date_range(g[TIME_COL].min(), g[TIME_COL].max(), freq=freq_alias)
        g = g.set_index(TIME_COL).reindex(idx)
        g.index.name = TIME_COL
        g[HOUSEHOLD_ID_COL] = customer_id
        g = g.reset_index()
        g = add_calendar_covariates(g, g[TIME_COL])
        for col in static_cols:
            g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
        for col in [TARGET_COL] + past_cov_cols + future_cov_cols:
            g[col] = pd.to_numeric(g[col], errors="coerce").fillna(0.0).astype("float32")
        if len(g) < INPUT_LEN + SEQ_LEN:
            continue
        out = g[[HOUSEHOLD_ID_COL, TIME_COL, TARGET_COL] + past_cov_cols + future_cov_cols].copy()
        out = out.rename(columns={HOUSEHOLD_ID_COL: "unique_id", TIME_COL: "ds", TARGET_COL: "y"})
        frames.append(out)
        row = {"unique_id": customer_id}
        for col in static_cols:
            row[col] = float(g[col].iloc[0])
        static_rows.append(row)

    if not frames:
        raise RuntimeError("No usable customer series were produced.")
    all_df = pd.concat(frames, ignore_index=True)
    static_df = pd.DataFrame(static_rows).drop_duplicates(subset=["unique_id"])
    train_ids = set(train_customers)
    val_ids = set(val_customers)
    train_df = all_df[all_df["unique_id"].isin(train_ids)].copy()
    val_df = all_df[all_df["unique_id"].isin(val_ids)].copy()
    static_train = static_df[static_df["unique_id"].isin(train_ids)].copy() if static_cols else None
    static_val = static_df[static_df["unique_id"].isin(val_ids)].copy() if static_cols else None
    return train_df, val_df, static_train, static_val, past_cov_cols, future_cov_cols, static_cols


train_df, val_df, static_train, static_val, past_cov_cols, future_cov_cols, static_cols = build_neuralforecast_frames()
print(f"Train rows={len(train_df)} | Val rows={len(val_df)}")
print(f"Train series={train_df['unique_id'].nunique()} | Val series={val_df['unique_id'].nunique()}")
print("Past covariates:", past_cov_cols)
print("Future covariates:", future_cov_cols)
print("Static covariates:", static_cols)



In [ ]:

torch.set_float32_matmul_precision("medium")
loss = MQLoss(quantiles=QUANTILES)
n_series = int(train_df["unique_id"].nunique())

model = iTransformer(
    h=SEQ_LEN,
    input_size=INPUT_LEN,
    n_series=n_series,
    hist_exog_list=past_cov_cols or None,
    futr_exog_list=future_cov_cols or None,
    stat_exog_list=static_cols or None,
    hidden_size=128,
    n_heads=2,
    e_layers=3,
    d_layers=1,
    d_ff=512,
    dropout=0.1,
    loss=loss,
    valid_loss=loss,
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    valid_batch_size=BATCH_SIZE,
    windows_batch_size=WINDOWS_BATCH_SIZE,
    inference_windows_batch_size=WINDOWS_BATCH_SIZE,
    scaler_type="standard",
    learning_rate=4e-3,
    random_seed=SEED,
    step_size=TRAIN_STRIDE,
    dataloader_kwargs=dataloader_kwargs(),
    alias="iTransformer",
    **trainer_kwargs(),
)
nf = NeuralForecast(models=[model], freq=pandas_freq(FREQ))
nf.fit(df=train_df, static_df=static_train, val_size=SEQ_LEN, verbose=True)



In [ ]:

# Chooses validation start positions without evaluating every possible overlapping window.
def eval_start_positions(series_length):
    max_start = series_length - SEQ_LEN
    if max_start < INPUT_LEN:
        return []
    return list(range(INPUT_LEN, max_start + 1, SEQ_LEN))


def select_prediction_columns(pred):
    candidates = [c for c in pred.columns if c not in {"unique_id", "ds", "cutoff"}]
    lower = {c.lower(): c for c in candidates}
    def find_any(patterns):
        for pattern in patterns:
            for low, original in lower.items():
                if pattern in low:
                    return original
        return None
    q50_col = find_any(["median", "q50", "0.5", "0.50"]) or find_any(["itransformer"])
    q10_col = find_any(["lo-80", "q10", "0.1", "0.10"])
    q90_col = find_any(["hi-80", "q90", "0.9", "0.90"])
    if q50_col is None:
        q50_col = candidates[0]
    if q10_col is None:
        q10_col = q50_col
    if q90_col is None:
        q90_col = q50_col
    return q10_col, q50_col, q90_col


# Runs validation forecasts for NeuralForecast iTransformer and exports the same plot/metric format.
def evaluate_itransformer(eval_pool):
    if eval_pool == "train":
        eval_df = train_df
        eval_static = static_train
        seed_offset = 1000
    elif eval_pool == "validation":
        eval_df = val_df
        eval_static = static_val
        seed_offset = 2000
    else:
        raise ValueError("eval_pool must be 'train' or 'validation'")

    y_real, q10s, q50s, q90s = [], [], [], []
    rng_eval = np.random.default_rng(SEED + seed_offset)
    ids = np.array(sorted(eval_df["unique_id"].unique()))
    rng_eval.shuffle(ids)
    for customer_id in ids:
        if len(y_real) >= EVAL_WINDOWS:
            break
        g = eval_df[eval_df["unique_id"] == customer_id].sort_values("ds").reset_index(drop=True)
        starts = eval_start_positions(len(g))
        if not starts:
            continue
        rng_eval.shuffle(starts)
        for forecast_start in starts:
            if len(y_real) >= EVAL_WINDOWS:
                break
            history = g.iloc[:forecast_start].copy()
            truth = g.iloc[forecast_start:forecast_start + SEQ_LEN].copy()
            if len(history) < INPUT_LEN or len(truth) != SEQ_LEN:
                continue
            futr_df = truth[["unique_id", "ds"] + future_cov_cols].copy() if future_cov_cols else None
            static_one = None
            if eval_static is not None:
                static_one = eval_static[eval_static["unique_id"] == customer_id].copy()
            pred = nf.predict(
                df=history,
                static_df=static_one,
                futr_df=futr_df,
                quantiles=QUANTILES,
                h=SEQ_LEN,
                verbose=False,
            )
            pred = pred.sort_values("ds").reset_index(drop=True)
            if len(pred) < SEQ_LEN:
                continue
            q10_col, q50_col, q90_col = select_prediction_columns(pred)
            real = truth["y"].to_numpy(dtype="float32").reshape(SEQ_LEN, 1)
            q10 = pred[q10_col].to_numpy(dtype="float32")[:SEQ_LEN].reshape(SEQ_LEN, 1)
            q50 = pred[q50_col].to_numpy(dtype="float32")[:SEQ_LEN].reshape(SEQ_LEN, 1)
            q90 = pred[q90_col].to_numpy(dtype="float32")[:SEQ_LEN].reshape(SEQ_LEN, 1)
            y_real.append(np.maximum(real, 0.0))
            q10s.append(np.maximum(np.minimum(q10, q90), 0.0))
            q50s.append(np.maximum(q50, 0.0))
            q90s.append(np.maximum(np.maximum(q10, q90), 0.0))
    if not y_real:
        raise RuntimeError(f"No {eval_pool} forecasts were produced. Try reducing INPUT_LEN or checking series length.")
    return np.stack(y_real, axis=0), np.stack(q10s, axis=0), np.stack(q50s, axis=0), np.stack(q90s, axis=0)


checkpoint_name = (
    f"baseline-{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__seq-{SEQ_LEN}"
    f"__seed-{SEED}__steps-{MAX_STEPS}__bs-{BATCH_SIZE}__in-{INPUT_LEN}"
    f"__stride-{TRAIN_STRIDE}__q-10-50-90"
)
checkpoint_dir = MODEL_DIR / "checkpoints" / checkpoint_name
checkpoint_dir.mkdir(parents=True, exist_ok=True)
nf.save(str(checkpoint_dir), overwrite=True, save_dataset=False)
print("Saved:", checkpoint_dir)

config = {
    "FREQ": FREQ,
    "PRIMARY_HORIZON": PRIMARY_HORIZON,
    "SEQ_LEN": SEQ_LEN,
    "INPUT_LEN": INPUT_LEN,
    "CONTEXT_HORIZONS": {k: str(v) for k, v in CONTEXT_HORIZONS.items()},
    "TRAIN_STRIDE": TRAIN_STRIDE,
    "NUM_LOADER_WORKERS": NUM_LOADER_WORKERS,
    "SEED": SEED,
    "VAL_FRAC": VAL_FRAC,
    "BATCH_SIZE": BATCH_SIZE,
    "MAX_STEPS": MAX_STEPS,
    "WINDOWS_BATCH_SIZE": WINDOWS_BATCH_SIZE,
    "CHECKPOINT_PATH": str(checkpoint_dir),
    "SPLIT_METHOD": "held_out_households",
    "PAST_COVARIATES": past_cov_cols,
    "FUTURE_COVARIATES": future_cov_cols,
    "STATIC_COVARIATES": static_cols,
}

for eval_pool in ("train", "validation"):
    y_real_kwh, y_q10_kwh, y_q50_kwh, y_q90_kwh = evaluate_itransformer(eval_pool)
    pool_config = dict(config)
    pool_config.update({
        "EVAL_POOL": eval_pool,
        "EVAL_WINDOWS": int(y_real_kwh.shape[0]),
        "REQUESTED_EVAL_WINDOWS": EVAL_WINDOWS,
        "selected_train_windows": int(y_real_kwh.shape[0]) if eval_pool == "train" else 0,
        "selected_val_windows": int(y_real_kwh.shape[0]) if eval_pool == "validation" else 0,
    })
    export_dir, metrics_path, plot_paths, metrics = export_results(
        checkpoint_name,
        pool_config,
        y_real_kwh,
        y_q10_kwh,
        y_q50_kwh,
        y_q90_kwh,
        MODEL_DIR / "evaluation_exports" / f"{eval_pool}_evaluation_exports",
    )
